# PilotNet from scratch

This notebook independently rebuilds the PilotNet model from Bojarski et al. (2016). It does not import the repository package: every model, dataset, training, and validation component is defined in the cells below.

## 1. Task and input contract

PilotNet is behavioral cloning: learn a scalar steering target from a front-camera image. The network receives RGB tensors with shape `(batch, 3, 66, 200)` and values in `[0, 1]`. The fixed geometry is important because the fully connected head expects a `64 x 1 x 18` convolutional output.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## 2. Input normalization

The first operation recenters pixel values around zero. This is intentionally a module, rather than hidden preprocessing, so the model has the same input contract during training and inference.

In [ ]:
class InputNormalization(nn.Module):
    def forward(self, images):
        if images.ndim != 4 or tuple(images.shape[1:]) != (3, 66, 200):
            raise ValueError(f'Expected (batch, 3, 66, 200), got {tuple(images.shape)}')
        return images - 0.5

normalization = InputNormalization()
normalization(torch.rand(2, 3, 66, 200)).shape

## 3. Convolutional feature extractor

Three `5x5`, stride-2 convolutions downsample the image while increasing channels from 24 to 48. Two `3x3`, stride-1 convolutions produce 64-channel features. ELU follows every convolution. For the paper input, spatial dimensions evolve as `66x200 -> 31x98 -> 14x47 -> 5x22 -> 3x20 -> 1x18`.

In [ ]:
class ConvolutionalFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(24, 36, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(36, 48, kernel_size=5, stride=2), nn.ELU(),
            nn.Conv2d(48, 64, kernel_size=3), nn.ELU(),
            nn.Conv2d(64, 64, kernel_size=3), nn.ELU(),
        )

    def forward(self, images):
        return self.layers(images)

features = ConvolutionalFeatures()
features(normalization(torch.rand(2, 3, 66, 200))).shape

## 4. Fully connected steering regressor

The final feature map contains `64 * 1 * 18 = 1,152` values. PilotNet maps these through widths `100 -> 50 -> 10 -> 1`. The final layer is linear because steering is a signed continuous target.

In [ ]:
class SteeringRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 1 * 18, 100), nn.ELU(),
            nn.Linear(100, 50), nn.ELU(),
            nn.Linear(50, 10), nn.ELU(),
            nn.Linear(10, 1),
        )

    def forward(self, feature_map):
        return self.layers(feature_map).squeeze(-1)

## 5. Assemble the policy and trace shapes

The complete policy composes the three modules without any hidden operations. The tracing cell runs each convolution in sequence, making the feature shape required by the regressor observable rather than assumed.

In [ ]:
class PilotNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.normalize = InputNormalization()
        self.features = ConvolutionalFeatures()
        self.regressor = SteeringRegressor()

    def forward(self, images):
        return self.regressor(self.features(self.normalize(images)))

model = PilotNet().to(device)
example = torch.rand(2, 3, 66, 200, device=device)
activation = model.normalize(example)
for layer in model.features.layers:
    activation = layer(activation)
    if isinstance(layer, nn.Conv2d):
        print(f'{layer}: {tuple(activation.shape)}')

print('Steering shape:', tuple(model(example).shape))
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))